In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os

# Tải và giải nén Micromamba
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba

# Thiết lập đường dẫn hệ thống
os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba-root"
os.environ["PATH"] = f"{os.getcwd()}/bin:" + os.environ["PATH"]

# Khởi tạo root prefix
!./bin/micromamba shell init -s bash -p /content/micromamba-root
!./bin/micromamba --version


bin/micromamba
The following argument was not expected: -p
Run with --help for more information.
2.5.0


In [ ]:

!./bin/micromamba create -p /content/spmamba-env python=3.10 --yes -q


!./bin/micromamba run -p /content/spmamba-env \
    pip install "setuptools<70" typeguard==2.13.3 h5py \
    pytorch-lightning==2.0.2 einops soundfile librosa \
    fast-bss-eval speechbrain==0.5.14 hydra-core rich \
    torch-complex packaging -q
!./bin/micromamba run -p /content/spmamba-env \
    pip install torch-mir-eval torch-optimizer -q
!./bin/micromamba run -p /content/spmamba-env pip install transformers -q


!./bin/micromamba run -p /content/spmamba-env \
    pip install torch==2.5.0 torchaudio==2.5.0 torchvision==0.20.0 \
    --index-url https://download.pytorch.org/whl/cu124 \
    --force-reinstall -q

!./bin/micromamba run -p /content/spmamba-env \
    pip install \
    "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.0/causal_conv1d-1.6.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl" \
    "https://github.com/state-spaces/mamba/releases/download/v2.3.0/mamba_ssm-2.3.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl" \
    --no-deps --force-reinstall -q

# Verify
!./bin/micromamba run -p /content/spmamba-env python -c "import torch; print('torch:', torch.__version__)"
!./bin/micromamba run -p /content/spmamba-env python -c "from mamba_ssm.modules.mamba_simple import Mamba; print('mamba-ssm: OK ✅')"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
torch: 2.5.0+cu124
mamba-ssm: OK ✅


In [ ]:
import os

REPO_DIR = "/content/spmamba"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hoanganh0106/spmamba.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 474 bytes | 474.00 KiB/s, done.
From https://github.com/hoanganh0106/spmamba
   1b72760..b828237  main       -> origin/main
Updating 1b72760..b828237
error: Your local changes to the following files would be overwritten by merge:
	audio_train.py
Please commit your changes or stash them before you merge.
Aborting
/content/spmamba


In [ ]:
# Copy từ Drive sang ổ cứng SSD của máy ảo
!cp /content/drive/MyDrive/data_30h.h5 /content/data.h5

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
/content/bin/micromamba run -p /content/spmamba-env \
    python audio_train.py --conf_dir configs/spmamba-3speakers.yml \
    --checkpoint_dir /content/drive/MyDrive/SPMamba_checkpoints/SPMamba-3Speakers \
    --resume

Instantiating datamodule <H5DataModule>
[H5DataModule] Split: train=22950, val=2700, test=1350
[H5Dataset] Loaded 22950 samples from /content/data.h5
  Sample rate: 16000 Hz, Speakers: 3
  Segment: 4.0s (64000 samples)
[H5Dataset] Loaded 2700 samples from /content/data.h5
  Sample rate: 16000 Hz, Speakers: 3
  Segment: 4.0s (64000 samples)
[H5Dataset] Loaded 1350 samples from /content/data.h5
  Sample rate: 16000 Hz, Speakers: 3
  Segment: full length (test mode)
Instantiating AudioNet <SPMamba>
Instantiating Optimizer <adam>
Instantiating Scheduler <ReduceLROnPlateau>
[Checkpoint Dir] /content/drive/MyDrive/SPMamba_checkpoints/SPMamba-3Speakers
[Checkpoint Dir exists] True
Instantiating Loss, Train <pairwise_neg_snr>, Val <pairwise_neg_sisdr>
Instantiating System <AudioLightningModule>
Instantiating ModelCheckpoint
Resuming from checkpoint: 
/content/drive/MyDrive/SPMamba_checkpoints/SPMamba-3Speakers/last-v1.ckpt
[Cleanup] Removed 
/content/drive/MyDrive/SPMamba_checkpoints/SPMamba-3

In [ ]:
import os

# Sửa trực tiếp file audio_train.py để fix lỗi monitor=None
train_script = '/content/spmamba/audio_train.py'
with open(train_script, 'r') as f:
    content = f.read()

# Thay thế đoạn khởi tạo ModelCheckpoint để đảm bảo luôn có monitor
old_code = 'step_checkpoint = ModelCheckpoint('
new_code = "monitor_val = arg_dic.get('checkpoint', {}).get('monitor', 'val_loss/dataloader_idx_0')\n    step_checkpoint = ModelCheckpoint(monitor=monitor_val, "

if old_code in content and 'monitor=monitor_val' not in content:
    content = content.replace(old_code, new_code)
    with open(train_script, 'w') as f:
        f.write(content)
    print("Đã sửa file audio_train.py để tự động nhận monitor.")
else:
    print("File đã được sửa hoặc không tìm thấy vị trí cần sửa.")

# Chạy lại lệnh huấn luyện
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
/content/bin/micromamba run -p /content/spmamba-env \
    python audio_train.py --conf_dir configs/spmamba-3speakers.yml \
    --checkpoint_dir /content/drive/MyDrive/SPMamba_checkpoints/SPMamba-3Speakers \
    --resume

File đã được sửa hoặc không tìm thấy vị trí cần sửa.
Instantiating datamodule <H5DataModule>
[H5DataModule] Split: train=22950, val=2700, test=1350
[H5Dataset] Loaded 22950 samples from /content/data.h5
  Sample rate: 16000 Hz, Speakers: 3
  Segment: 4.0s (64000 samples)
[H5Dataset] Loaded 2700 samples from /content/data.h5
  Sample rate: 16000 Hz, Speakers: 3
  Segment: 4.0s (64000 samples)
[H5Dataset] Loaded 1350 samples from /content/data.h5
  Sample rate: 16000 Hz, Speakers: 3
  Segment: full length (test mode)
Instantiating AudioNet <SPMamba>
Instantiating Optimizer <adam>
Instantiating Scheduler <ReduceLROnPlateau>
[Checkpoint Dir] /content/drive/MyDrive/SPMamba_checkpoints/SPMamba-3Speakers
[Checkpoint Dir exists] True
Instantiating Loss, Train <pairwise_neg_snr>, Val <pairwise_neg_sisdr>
Instantiating System <AudioLightningModule>
Instantiating ModelCheckpoint
Resuming from checkpoint: 
/content/drive/MyDrive/SPMamba_checkpoints/SPMamba-3Speakers/last.ckpt
[Cleanup] Skipped (re